# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdelrhman-Moubarak/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup

In [15]:
import os
import duckdb

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    token = os.environ.get("HF_TOKEN")

if not token:
    raise RuntimeError("HF_TOKEN not found. Set it as a Colab Secret or in your local .env file.")

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"""
    CREATE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{token}'
    );
""")

BASE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"{BASE}/dim_clients.parquet"
DIM_CONTENT = f"{BASE}/dim_content.parquet"
FACT_MARCH  = f"{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet"

print("DuckDB ready with HF secret configured.")

DuckDB ready with HF secret configured.


## 1. Unit of analysis + time window

Each row is one content item's H1 (March 1-15, 2026) performance summary, aggregated from
`fact_content_daily_performance` for `month=2026-03`. Since this is my first time working
with a dataset set up this way, I wanted to stick closely to the point-in-time snapshot I'm
allowed to know about before deciding whether a piece of content looks like a refresh
candidate.

**Time window:** March 2026 only, split into two halves - H1 (Mar 1-15) as my feature
window, H2 (Mar 16-31) as the observed outcome window. I made sure the final month (June
2026) is not queried anywhere in this notebook, they stay sealed.

In [16]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{FACT_MARCH}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()
print(len(grain_check))

h1_span = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{FACT_MARCH}')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
""").df()
h1_span

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

0


,row_count,min_date,max_date
0,4642255,2026-03-01,2026-03-15


## 2. Fields: feature / label / context / excluded

**Feature** (measured or fixed before the H2 outcome window, decision-support inputs):
- `avg_position_h1` → average `gsc_avg_position` over H1
- `total_clicks_h1` → summed `gsc_clicks` over H1
- `avg_engagement_rate_h1` → `ga4_engaged_sessions / ga4_sessions`, averaged over H1, only where `ga4_data_available IS TRUE`
- `content_age_days` → days between `dim_content.content_created_date` and March 1
- `content_type` → static category from `dim_content`

**Label** (what I'm trying to predict, built from H2 only, never used as a feature):
- `is_declining` → 1 if H2 `total_clicks` comes out lower than H1 `total_clicks_h1`, else 0

**Context** (only for joining, grouping, or reading, never fed into a model):
- `content_hash_id`, `client_hash_id` → pseudonymous IDs

**Excluded:**
- `fact_content_query_90d`: I left this one out because its fixed 90-day window doesn't line
  up with the H1/H2 split I'm using here. Since I'm still new to this, I didn't want to risk
  building a feature I couldn't actually verify as H1-only, so rather than guess I just
  excluded it entirely to avoid an unverified leak.
- `last_optimized_date` / `optimization_eligible_date` (in `dim_content`): these do seem
  genuinely relevant to refresh scoring, but I kept them out of this notebook for now to keep
  the contract scoped to only what I could verify below. Flagging them here as candidates to
  revisit in a later week.

In [17]:
FEATURE_COLS = [
    "avg_position_h1", "total_clicks_h1", "avg_engagement_rate_h1",
    "content_age_days", "content_type",
]
LABEL_COL = "is_declining"
CONTEXT_COLS = ["content_hash_id", "client_hash_id"]
EXCLUDED = ["fact_content_query_90d table", "last_optimized_date", "optimization_eligible_date"]

id_check = con.sql(f"""
    SELECT
        COUNT(DISTINCT content_hash_id) AS distinct_content_ids,
        COUNT(DISTINCT client_hash_id)  AS distinct_client_ids,
        COUNT(*)                        AS total_rows
    FROM read_parquet('{FACT_MARCH}')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
""").df()
print(FEATURE_COLS)
print(LABEL_COL)
print(CONTEXT_COLS)
print(EXCLUDED)
id_check

['avg_position_h1', 'total_clicks_h1', 'avg_engagement_rate_h1', 'content_age_days', 'content_type']
is_declining
['content_hash_id', 'client_hash_id']
['fact_content_query_90d table', 'last_optimized_date', 'optimization_eligible_date']


,distinct_content_ids,distinct_client_ids,total_rows
0,319759,52,4642255


## 3. Verify it with queries (grain, counts, missing values, windows)

Grain and window were already verified in section 1, so this section covers what's left:
missingness, then the feature/label build, then the leakage proof. Concretely, here's what
the code below does, in order, on `month=2026-03` only:

1. **Availability**: I count how much GA4 data actually exists in the H1 slice, filtered with
   `IS TRUE` (the flag can be NULL, not just TRUE/FALSE, so using `= TRUE` alone would
   silently miscount things, and since I'm still learning this dataset I wanted to double
   check that before trusting anything downstream).
2. **Build the five features**: I aggregate H1 (Mar 1-15) only, per content item, into
   `avg_position_h1`, `total_clicks_h1`, `avg_engagement_rate_h1`, `content_age_days`,
   `content_type`. Each one is honestly knowable before H2 even exists.
3. **Build the label from H2**: a completely separate query over Mar 16-31, computing
   `is_declining`. I kept this apart from step 2 on purpose, so H2 data can't accidentally
   leak into an H1 feature.
4. **The real score**: a small logistic regression trained on the 5 real features only.
   This is the number that actually means something.
5. **The deliberate leak**: I add one H2-derived column (`avg_position_h2`) as if it were a
   6th feature, just to see what happens. The score jumps toward a suspiciously high number,
   which proves the model is reading the answer rather than learning a pattern. I then drop
   that column and keep the honest score from step 4.

In [18]:
# Availability check

availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS available_rows
    FROM read_parquet('{FACT_MARCH}')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
""").df()
availability["pct_available"] = (availability["available_rows"] / availability["total_rows"] * 100).round(1)
print("====== Availability check ======")
print(availability)


# Build the five features from H1 only (Mar 1-15)

features_h1 = con.sql(f"""
    WITH h1 AS (
        SELECT
            content_hash_id, client_hash_id,
            AVG(gsc_avg_position) AS avg_position_h1,
            SUM(gsc_clicks)       AS total_clicks_h1,
            AVG(CASE WHEN ga4_data_available IS TRUE
                     THEN CAST(ga4_engaged_sessions AS DOUBLE) / NULLIF(ga4_sessions, 0)
                END) AS avg_engagement_rate_h1
        FROM read_parquet('{FACT_MARCH}')
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        h1.*,
        DATE_DIFF('day', dc.content_created_date, DATE '2026-03-01') AS content_age_days,
        dc.content_type
    FROM h1
    JOIN read_parquet('{DIM_CONTENT}') dc USING (content_hash_id)
""").df()
print("====== features from H1 ======")
print(features_h1.shape)
print(features_h1.head())


# Build the label from H2 only (Mar 16-31), then join

label_h2 = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id,
        SUM(gsc_clicks)       AS total_clicks_h2,
        AVG(gsc_avg_position) AS avg_position_h2
    FROM read_parquet('{FACT_MARCH}')
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
    GROUP BY content_hash_id, client_hash_id
""").df()

frame = features_h1.merge(label_h2, on=["content_hash_id", "client_hash_id"], how="inner")
frame["is_declining"] = (frame["total_clicks_h2"] < frame["total_clicks_h1"]).astype(int)
print("====== label from H2 ======")
print(frame["is_declining"].value_counts())


# The real score, 5 real features only


from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

frame_enc = frame.copy()
frame_enc["content_type_code"] = frame_enc["content_type"].astype("category").cat.codes
real_cols = ["avg_position_h1", "total_clicks_h1", "avg_engagement_rate_h1",
               "content_age_days", "content_type_code"]

frame_enc = frame_enc.dropna(subset=real_cols + ["is_declining"])
X, y = frame_enc[real_cols], frame_enc["is_declining"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

real_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
real_score = accuracy_score(y_te, real_model.predict(X_te))
print("Real Score:", real_score)


#The deliberate leak, then remove it

leaky_cols = real_cols + ["avg_position_h2"]
frame_leak = frame_enc.dropna(subset=leaky_cols + ["is_declining"])
Xl, yl = frame_leak[leaky_cols], frame_leak["is_declining"]
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xl, yl, test_size=0.3, random_state=42, stratify=yl)

leaky_model = LogisticRegression(max_iter=1000).fit(Xl_tr, yl_tr)
leaky_score = accuracy_score(yl_te, leaky_model.predict(Xl_te))
print("====== The deliberate leak ======")
print("Leaky score: ", leaky_score, "Real Score: ", real_score)

print("Real Score:", real_score)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

====== Availability check ======
   total_rows  available_rows  pct_available
0     4642255        159060.0            3.4


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

====== features from H1 ======
(319759, 7)
            content_hash_id           client_hash_id  avg_position_h1  \
0  content_67741cce996cfafa  client_62f4a7e64f5e0096         4.638889   
1  content_2e6360ad20fd7107  client_62f4a7e64f5e0096         3.737399   
2  content_65c50dfe9d87a585  client_62f4a7e64f5e0096         6.156643   
3  content_275b6f7f733016d4  client_62f4a7e64f5e0096         4.449176   
4  content_4dc944b7d0b65ecc  client_62f4a7e64f5e0096         4.449490   

   total_clicks_h1  avg_engagement_rate_h1  content_age_days     content_type  
0              1.0                     NaN                17  keyword article  
1              1.0                     NaN                17  keyword article  
2              0.0                     NaN                17  keyword article  
3              1.0                     NaN                17  keyword article  
4              0.0                     NaN                17  keyword article  


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

====== label from H2 ======
is_declining
0    290769
1     28989
Name: count, dtype: int64
Real Score: 0.6556962025316456
====== The deliberate leak ======
Leaky score:  0.651551724137931 Real Score:  0.6556962025316456
Real Score: 0.6556962025316456


## 4. Data limits

History depth varies by client (`dim_clients.gsc_data_start`). Some clients joined more
recently, so their content might have little or no data before March, meaning "H1" isn't
really a stable pattern for them, it might just be close to their entire available history.
That means this slice leans more toward long-established clients, and any decline signal I'm
seeing here is only directional at best for the newly onboarded ones, it might not
generalize.

In [19]:
history_check = con.sql(f"""
    SELECT
        dcl.gsc_data_start,
        COUNT(DISTINCT f.content_hash_id) AS content_items
    FROM read_parquet('{FACT_MARCH}') f
    JOIN read_parquet('{DIM_CLIENTS}') dcl USING (client_hash_id)
    WHERE f.report_date BETWEEN '2026-03-01' AND '2026-03-15'
    GROUP BY dcl.gsc_data_start
    ORDER BY dcl.gsc_data_start DESC
    LIMIT 10
""").df()
print("Most recent client history-start dates observed, with content item counts:")
history_check

Most recent client history-start dates observed, with content item counts:


,gsc_data_start,content_items
0,2026-03-12,611
1,2026-03-11,276
2,2026-02-26,239
3,2026-02-19,41154
4,2026-02-17,416
5,2026-01-09,5878
6,2025-12-17,5574
7,2025-11-17,200
8,2025-11-16,5457
9,2025-11-15,9308


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.